<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Open Source Functions(td_lightgbm) in Vantage
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style = 'font-size:20px;font-family:Arial'><b>Introduction</b></p>
<p style = 'font-size:16px;font-family:Arial'> The Teradata Python Package introduces Open-Source machine functions, which exposes most of the functionality of open-source packages like scikit-learn. With teradataml OpenSourceML, users can run these functions without needing to pull the data to their client. It offers a simple interface object for the open-source machine learning functions, allowing them to be used with the same syntax and arguments as the actual open-source functions and classes.<br>Functions/classes from opensource packages generates single model, i.e., model is trained on all the data. Unlike traditional opensource packages, teradataml OpenSourceML allows user to generate distributed models a.k.a. multiple models a.k.a. micro models. With the MPP architecture that Teradata Vantage provides, teradataml OpenSourceML can tap, process and solve large set of use cases where distributed models are needed. To enable this support, teradataml OpenSourceML introduces another argument partition_columns, applicable for all functions, that accepts the column to be used to partition the data and generate the models for partitioned data.<br>
<ul style = 'font-size:16px;font-family:Arial'>Supported Open-Source Packages:<li>scikit-learn</li>
    <li>lightGBM</li>



<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>1. Initiate a connection to Vantage</b>

<p style = 'font-size:16px;font-family:Arial'>In the section, we import the required libraries and set environment variables and environment paths (if required).

In [ ]:
from teradataml import *

#from teradatasqlalchemy.types import *

#from teradataml import to_numeric
# Modify the following to match the specific client environment settings
display.max_rows = 5

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial'><b>1.1 Connect to Vantage</b></p>
<p style = 'font-size:16px;font-family:Arial'>You will be prompted to provide the password. Enter your password, press the Enter key, and then use the down arrow to go to the next cell.</p>

In [ ]:
%run -i ../../UseCases/startup.ipynb
eng = create_context(host = 'host.docker.internal', username='demo_user', password = password)
print(eng)

In [ ]:
%%capture
execute_sql('''SET query_band='DEMO=PP_OpenSourceML_td_lightgbm.ipynb;' UPDATE FOR SESSION; ''')

<p style = 'font-size:16px;font-family:Arial'>Begin running steps with Shift + Enter keys. </p>

<hr style='height:1px;border:none;background-color:#00233C;'>

<p style = 'font-size:18px;font-family:Arial;color:#00233c'><b>1.2 Getting Data for This Demo</b></p>

<p style = 'font-size:16px;font-family:Arial;color:#00233C'>Here, we will get the model sample input data which is available in the teradataml library and use the same to show the usage of the function.</p>

In [ ]:
load_example_data("openml", ["multi_model_classification", "multi_model_regression"])

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial;'>2. Data Exploration</b>
<p style = 'font-size:16px;font-family:Arial;'>Create a "Virtual DataFrame" that points to the data set in Vantage. Check the shape of the dataframe as check the datatype of all the columns of the dataframe.</p>

In [ ]:
## Classification data.
# Create DataFrames.
df_train_classif = DataFrame('multi_model_classification')
df_train_classif

In [ ]:
# Create required classification DataFrames.
df_x_classif = df_train_classif.select(["col1", "col2", "col3", "col4"])
df_y_classif = df_train_classif.select("label")

In [ ]:
## Regression data.
# Create DataFrames.
df_train_reg = DataFrame('multi_model_regression')
df_train_reg

In [ ]:
# Create required regression DataFrames.
df_x_reg = df_train_reg.select(["col1", "col2", "col3", "col4"])
df_y_reg = df_train_reg.select("label")

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>3. Model Training</b>
<p style = 'font-size:18px;font-family:Arial'><b>3.1 Single Model Training</b>
<p style = 'font-size:16px;font-family:Arial'>Once required teradataml DataFrames are created we need to create Dataset objects.

In [ ]:
# Creating Dataset Object.
obj_s1 = td_lightgbm.Dataset(df_x_classif, df_y_classif)
obj_s1

<p style = 'font-size:16px;font-family:Arial'>After creating Dataset object we run training function with record_evaluation and early_stopping callbacks

In [ ]:
rec = {} # To pass this empty dictionary to record_evaluation callback.
# Training With valid_sets and callbacks argument.
opt_tr_s = td_lightgbm.train(params={}, train_set=obj_s1, num_boost_round=30,
                                 callbacks=[td_lightgbm.record_evaluation(rec), td_lightgbm.early_stopping(3)],
                                 valid_sets=[obj_s1])
opt_tr_s

<p style="font-size:16px;font-family:Arial">Similar to actual train() function of lightgbm, td_lightgbm also displays console output and returns Booster object. However, this Booster object is not lightgbm’s Booster object but it is teradata opensource ml's internal Booster wrapper object.Below is the differences between training functionality of lightgbm and td_lightgbm.
    <ul style="font-size:16px;font-family:Arial">
        <li>lightgbm populates the record evaluation results in the variable that we passed to the record_evaluation function.</li>
        <li>td_lightgbm provides an attribute record_evaluation_result for the object returned by train() method, which can be accessed as below</li>


In [ ]:
opt_tr_s.record_evaluation_result

<hr style="height:1px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>3.2 Single model Cross Validation</b>
<p style = 'font-size:16px;font-family:Arial'>From below we can see how we can run single model cross validation using td_lightgbm.


In [ ]:
# Training without return_cvbooster argument.
opt_cv_s = td_lightgbm.cv(params={}, train_set=obj_s1, callbacks=[td_lightgbm.early_stopping(5)])
opt_cv_s

<p style = 'font-size:16px;font-family:Arial'>The argument return_cvbooster is not supported yet for cv().

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial;'><b>3.3 Multi model Training</b></p>
<p style = 'font-size:16px;font-family:Arial;'>After teradataml DataFrames are created we create required Dataset objects.In below code we have created two Dataset objects, one for training and another for validation. Note that we have used partition_columns argument for distributed/multi model training.
<br>Note:
<br> •The teradataml DataFrames that are passed should be created from the same parent teradataml DataFrame using select() API.<br>•The partition columns should be present in the parent DataFrame from which required DataFrames are derived from.

In [ ]:
# Training Dataset.
obj_m = td_lightgbm.Dataset(df_x_classif, df_y_classif,
                                partition_columns=["partition_column_1", "partition_column_2"])
obj_m

In [ ]:
# Validation dataset.
obj_m_v = td_lightgbm.Dataset(df_x_classif, df_y_classif,
                                  partition_columns=["partition_column_1", "partition_column_2"])
obj_m_v

<p style = 'font-size:16px;font-family:Arial;'>After creating Dataset objects we run training function with record_evaluation callback.

In [ ]:
# Training with valid_sets and callbacks argument.
rec = {}
opt_tr_m = td_lightgbm.train(params={}, train_set=obj_m_v, num_boost_round=30,
                                 callbacks=[td_lightgbm.record_evaluation(rec)],
                                 valid_sets=[obj_m_v, obj_m_v])
opt_tr_m

<p style = 'font-size:16px;font-family:Arial;'>Since there are multiple models involved, each trained model have different console output and record_evaluation result. All these are provided in pandas Dataframe and can be accessed using model_info attribute. User can access pandas Dataframe and individual record evaluation results or console outputs as below.


In [ ]:
opt_tr_m.model_info

In [ ]:
# console output.
print(opt_tr_m.model_info.iloc[0]["console_output"])

In [ ]:
# record_evaluation result.
print(opt_tr_m.model_info.iloc[0]["record_evaluation_result"])
{'valid_0': OrderedDict([('l2', [0.21963737683498566, 0.1965251124298607, ..., 0.06526645509697025])]), 'valid_1': OrderedDict([('l2', [0.21963737683498566, 0.1965251124298607, ..., 0.06526645509697025])])}

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial;'><b>3.4 Multi model Cross Validation</b></p>

<p style = 'font-size:16px;font-family:Arial;'>Below is an example of multi model cross validation using td_lightgbm.</p>

In [ ]:
# Training without return_cvbooster argument.
opt_cv_m = td_lightgbm.cv(params={}, train_set=obj_m_v, num_boost_round=30)
opt_cv_m

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial;'><b>3.5 Using Booster for prediction</b></p>

<p style = 'font-size:16px;font-family:Arial;'>Teradataml’s Booster Wrapper object is returned by train(), which can be used to predict values on new data. label argument is added to predict() by teradataml for the function to return label column in output DataFrame so that both true values and predicted values are in same teradataml DataFrame. Then resultant DataFrame can be used to get some evaluation metrics like accuracy_score using OpensourceML’s scikit-learn metrics functions.
    <br><b>Single model prediction</b><br>
To get predicted values, predict function should be called on trained Booster object.</p>

In [ ]:
opt_pred_s = opt_tr_s.predict(data=df_x_classif, label=df_y_classif, num_iteration=20)
opt_pred_s.head(3)

<p style = 'font-size:16px;font-family:Arial;'>All other supported functions can be accessed through Booster object. 

In [ ]:
# model_to_string
str_opt = opt_tr_s.model_to_string(-1)
print(str_opt)

In [ ]:
# model_from_string
opt2 = opt_tr_s.model_from_string(str_opt)
opt2

<p style = 'font-size:16px;font-family:Arial;'><b>Multi model prediction</b><br>
   For prediction in muti model case, partition_columns argument may or may not be provided. If this argument is not provided, the partition columnsand values are taken from training.

In [ ]:
opt_pred_m = opt_tr_m.predict(data=df_x_classif, label=df_y_classif, num_iteration=20)
opt_pred_m.head(3)

<p style = 'font-size:16px;font-family:Arial;'>Booster Wrapper object can be instantiated by passing arguments to the class as follows:

In [ ]:
# By passing "model_file" argument. For that, existing model needs to be saved to a file.
opt_tr_s.save_model("model_file_single_partition")

# Instantiate Booster Wrapper object through "model_file" argument.
direct_obj = td_lightgbm.Booster(model_file="model_file_single_partition")
direct_obj

In [ ]:
# Another way is to use "model_str" argument if model is already present as string.
direct_obj1 = td_lightgbm.Booster(model_str = str_opt)
direct_obj1

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>4. Running scikit-learn API</b>
<p style = 'font-size:16px;font-family:Arial'>Lightgbm offers 4 scikit-learn APIs whose syntax and usage are similar to scikit-learn classes. All 4 classes are supported for single and multi model cases.<ul style = 'font-size:16px;font-family:Arial'><li>lightgbm.LGBMModel</li><li>lightgbm.LGBMClassifier</li><li>lightgbm.LGBMRegressor</li><li>lightgbm.LGBMRanker</li>
</ul></p>
<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial;'><b>4.1 Single model case - LGBMRegressor</b></p>



In [ ]:
# Create LGBMRegressor object.
obj = td_lightgbm.LGBMRegressor(num_leaves=5, n_estimators=15,
                                    learning_rate=0.01)
obj

In [ ]:
# Set/update parameters.
obj.set_params(n_estimators=10)

In [ ]:
# Train the model.
obj.fit(df_x_reg, df_y_reg, callbacks=[td_lightgbm.log_evaluation()])

In [ ]:
# Predict the values.
obj.predict(X=df_x_reg)

In [ ]:
# Accessing some attributes.
obj.feature_importances_

In [ ]:
obj.objective_

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial;'><b>4.2 Multi-model case - LGBMClassifier</b></p>
<p style = 'font-size:16px;font-family:Arial;'>For multi-model training, fit() method should take partition_columns argument. And these columns should be present in parent DataFrame from which X and y teradataml DataFrames are derived from.


In [ ]:
# Create LGBMClassifier object.
obj = td_lightgbm.LGBMClassifier(num_leaves=5, n_estimators=15, learning_rate=0.01)
obj

In [ ]:
# Set/update parameters.
obj.set_params(n_estimators=10)

In [ ]:
# Train the model.
obj.fit(df_x_classif, df_y_classif, callbacks=[td_lightgbm.log_evaluation()],
            partition_columns=["partition_column_1", "partition_column_2"])

# Predict the values.
obj.predict(X=df_x_classif)


In [ ]:
# Score per partition.
obj.score(df_x_classif, df_y_classif, sample_weight=df_train_classif.select("group_column"))

In [ ]:
# Accessing some attributes.
obj.feature_importances_

In [ ]:
obj.objective_

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>5. Cleanup</b>

<p style = 'font-size:18px;font-family:Arial'><b>Work Tables</b></p>
<p style = 'font-size:16px;font-family:Arial'>The following code will clean up intermediate tables.</p>

In [ ]:
db_drop_table(table_name='multi_model_classification')

In [ ]:
db_drop_table(table_name='multi_model_regression')

In [ ]:
remove_context()

<hr style="height:1px;border:none;">

<p style = 'font-size:16px;font-family:Arial'><b>Links:</b></p>
<ul style = 'font-size:16px;font-family:Arial'>
    <li>Teradataml Python reference: <a href = 'https://docs.teradata.com/search/all?query=Python+Package+User+Guide&content-lang=en-US'>here</a></li>
    <li>td_lightgbm function reference: <a href = 'https://docs.teradata.com/search/all?query=td_lightgbm&content-lang=en-US'>here</a></li>
    </ul>

<footer style="padding-bottom:35px; border-bottom:3px solid #91A0Ab">
    <div style="float:left;margin-top:14px">ClearScape Analytics™</div>
    <div style="float:right;">
        <div style="float:left; margin-top:14px">
            Copyright © Teradata Corporation - 2026. All Rights Reserved
        </div>
    </div>
</footer>